In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from datasets import Dataset

# Load Model
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

# LoRA Config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Dataset
data = [
    {"text": "Q: What is the capital of Ethiopia?\nA: The capital of Ethiopia is Addis Ababa."},
    {"text": "Q: What is the national dish of Ethiopia?\nA: Injera is the national dish made from teff flour."},
    {"text": "Q: Where was coffee discovered?\nA: Coffee was discovered in Ethiopia."},
    {"text": "Q: Tell me about Lalibela.\nA: Lalibela is famous for its rock-hewn churches."},
    {"text": "Q: What is special about Ethiopian culture?\nA: Ethiopia has rich history, diverse ethnic groups, and unique traditions."},
]

dataset = Dataset.from_list(data)

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256, padding="max_length")

tokenized_dataset = dataset.map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

# Training
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/llm-portfolio",
    num_train_epochs=4,
    per_device_train_batch_size=4,
    learning_rate=3e-4,
    fp16=True,
    optim="adamw_8bit",
    report_to="none",
    save_strategy="no"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

# Save
model.save_pretrained("/content/drive/MyDrive/llm-portfolio")
tokenizer.save_pretrained("/content/drive/MyDrive/llm-portfolio")
print("✅ Fine-tuning completed!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Step,Training Loss


✅ Fine-tuning completed!


In [4]:
# Use an 'ls' command to check the contents of the adapter directory
!ls -F /content/drive/MyDrive/llm-portfolio

# If the directory doesn't exist, this will also indicate that.
# If it exists but 'adapter_config.json' is not listed, you need to ensure the adapter is saved correctly.

adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = "/content/drive/MyDrive/llm-portfolio"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, adapter_path)

def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(inputs.input_ids, max_new_tokens=300, temperature=0.7, top_p=0.9, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test
print(generate("You are an Ethiopian AI assistant. What is special about Ethiopia?"))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_si

You are an Ethiopian AI assistant. What is special about Ethiopia? 1. Highest number of ethnic groups in Africa: Ethiopia is home to 80 ethnic groups. 2. Richest country in Africa: Ethiopia is the richest country in Africa, with a per capita income of $1,626. 3. First country to use electricity: Ethiopia became the first country in Africa to use electricity in 1910. 4. First country to run a railway: Ethiopia became the first country in Africa to run a railway in 1896. 5. First country to use telegraph: In 1899, Ethiopia became the first country in Africa to use a telegraph to communicate with other countries. 6. First country to use telephones: In 1903, Ethiopia became the first country in Africa to use telephones. 7. First country to produce tea: In 1934, Ethiopia became the first country in Africa to produce tea. 8. First country to produce coffee: In 1936, Ethiopia became the first country in Africa to produce coffee. 9. First country to import oil: In 1956, Ethiopia became the fir